# Flat Files Docs

Access historical data in bulk via downloadable CSVs. If you need comprehensive datasets for extensive analysis, backtesting, or machine learning models, our Flat Files offer a great way to download large volumes of historical data.

In [1]:
aws_access_key_id = "b2efb6eb-09a9-470b-be7a-063ff3614bb2"
aws_secret_access_key = "gE_mClabKAWkIBuc36jStM5blbyqNUms"
s3_endpoint = "https://files.massive.com"
bucket_name = "flatfiles"

save_dir = "/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/data/minute_aggs"

In [2]:
import boto3
from botocore.config import Config

# Initialize a session using your credentials
session = boto3.Session(
  aws_access_key_id=aws_access_key_id,
  aws_secret_access_key=aws_secret_access_key,
)

In [3]:
# Create a client with your session and specify the endpoint
s3 = session.client(
  's3',
  endpoint_url=s3_endpoint,
  config=Config(signature_version='s3v4'),
)

In [4]:
# List Example
# Initialize a paginator for listing objects
paginator = s3.get_paginator('list_objects_v2')

# Choose the appropriate prefix depending on the data you need:
# - 'global_crypto' for global cryptocurrency data
# - 'global_forex' for global forex data
# - 'us_indices' for US indices data
# - 'us_options_opra' for US options (OPRA) data
# - 'us_stocks_sip' for US stocks (SIP) data
prefix = 'us_stocks_sip'  # Example: Change this prefix to match your data need

In [5]:
# List objects using the selected prefix
for page in paginator.paginate(Bucket='flatfiles', Prefix=prefix):
  for obj in page['Contents']:
    print(obj['Key'])

# Copy example
# Specify the bucket name
bucket_name = bucket_name

# Specify the S3 object key name
object_key = 'us_stocks_sip/minute_aggs_v1/2026/05/2026-05-01.csv.gz'

# Specify the local file name and path to save the downloaded file
# This splits the object_key string by '/' and takes the last segment as the file name
local_file_name = object_key.split('/')[-1]

# This constructs the full local file path
local_file_path = save_dir + '/' + local_file_name

# Download the file
# s3.download_file(bucket_name, object_key, local_file_path)


us_stocks_sip/day_aggs_v1/2003/09/2003-09-10.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-11.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-12.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-15.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-16.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-17.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-18.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-19.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-22.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-23.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-24.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-25.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-26.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-29.csv.gz
us_stocks_sip/day_aggs_v1/2003/09/2003-09-30.csv.gz
us_stocks_sip/day_aggs_v1/2003/10/2003-10-01.csv.gz
us_stocks_sip/day_aggs_v1/2003/10/2003-10-02.csv.gz
us_stocks_sip/day_aggs_v1/2003/10/2003-10-03.csv.gz
us_stocks_sip/day_aggs_v1/2003/10/2003-10-06.csv.gz
us_stocks_si

In [16]:
# load large dataframe by chunking

import pandas as pd
from typing import List
ticker_list = [
    "AAPL",
    "MU",
]

import pandas as pd

def convert_timestamp_to_us_datetime(timestamp: int, unit="ms"):
    """
    Converts a millisecond timestamp to US/Eastern time.

    NYSE/NASDAQ are located at US/Eastern
    """
    return pd.to_datetime(
        timestamp, unit=unit, utc=True
    ).tz_convert('US/Eastern')      
    
def load_csv_by_chunking(
    filepath: str, ticker_list: List[str],
    chunk_size = 100000
):

    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        filtered_chunk = chunk[chunk["ticker"].isin(ticker_list)]
        chunks.append(filtered_chunk)

    df = pd.concat(chunks)
    return df

In [20]:
filepath = "/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/data/minute_aggs/2026-05-01.csv.gz"
df = load_csv_by_chunking(filepath, ticker_list)
df["us_datetime"] = df["window_start"].apply(
    lambda x: convert_timestamp_to_us_datetime(x, unit="ns"))
df["us_date"] = df["us_datetime"].dt.date
df["hour"] = df["us_datetime"].dt.hour

In [22]:
agg_logic = {
    "open": "first",    # The first price of the hour
    "high": "max",      # The highest price in that hour
    "low": "min",       # The lowest price in that hour
    "close": "last",    # The final price of the hour
    "volume": "sum",    # Total volume traded in that hour
    "transactions": "sum" # Total count of trades in that hour
}

hour_df = df.groupby(
    ["ticker", "us_date", "hour"]
).agg(agg_logic).reset_index()

In [23]:
hour_df

,ticker,us_date,hour,open,high,low,close,volume,transactions
0,AAPL,2026-05-01,4,278.000000,280.2200,277.5600,278.7800,2.637000e+05,8500
1,AAPL,2026-05-01,5,278.880000,280.4800,278.1500,279.7400,1.274838e+05,3884
2,AAPL,2026-05-01,6,279.790000,280.0000,279.0300,279.3800,8.000364e+04,2828
3,AAPL,2026-05-01,7,279.230000,282.4200,279.0500,281.2375,8.009688e+05,16184
4,AAPL,2026-05-01,8,281.230000,281.4700,280.0100,280.0500,8.078497e+05,14573
5,AAPL,2026-05-01,9,280.050000,284.7500,271.3500,283.9500,1.795586e+07,278859
6,AAPL,2026-05-01,10,283.950000,287.2200,282.0100,282.6100,1.534336e+07,312619
7,AAPL,2026-05-01,11,282.650000,285.4900,281.4600,284.7500,7.996999e+06,193978
8,AAPL,2026-05-01,12,284.710000,285.0000,282.3951,283.1170,8.643411e+06,125762
9,AAPL,2026-05-01,13,283.130000,283.5000,281.4200,281.4750,4.288033e+06,87485


In [ ]:
# 1. Ensure us_datetime is the index
df = df.set_index("us_datetime")

# 2. Define the aggregation logic for each column
agg_logic = {
    "open": "first",    # The first price of the hour
    "high": "max",      # The highest price in that hour
    "low": "min",       # The lowest price in that hour
    "close": "last",    # The final price of the hour
    "volume": "sum",    # Total volume traded in that hour
    "transactions": "sum" # Total count of trades in that hour
}

# 3. Resample to 1 Hour ('H')
hourly_df = df.resample("H").agg(agg_logic).dropna()

# 4. Clean up
hourly_df = hourly_df.reset_index()

In [12]:
import pandas as pd

# Pandas handles decompression on the fly
filepath = "/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/data/2025-11-05.csv.gz"
filepath = "/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/data/2026-05-08.csv.gz"
filepath = "/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/data/minute_aggs/2026-05-01.csv.gz"
df = pd.read_csv(filepath, nrows=500)

In [14]:
stock = ["ICHR", "POET"] 

,ticker,volume,open,close,high,low,window_start,transactions,us_datetime
0,A,10805.484156,116.130,115.665,116.275,115.665,1777642200000000000,74,2026-05-01 09:30:00-04:00
1,A,1617.024991,115.270,115.755,115.920,115.270,1777642260000000000,23,2026-05-01 09:31:00-04:00
2,A,8080.000000,115.705,115.685,115.705,115.685,1777642320000000000,34,2026-05-01 09:32:00-04:00
3,A,7933.137804,115.705,115.390,115.705,115.390,1777642380000000000,62,2026-05-01 09:33:00-04:00
4,A,2832.000000,115.390,115.535,115.535,115.235,1777642440000000000,47,2026-05-01 09:34:00-04:00
...,...,...,...,...,...,...,...,...,...
495,AA,3664.216000,63.320,63.320,63.340,63.300,1777648620000000000,66,2026-05-01 11:17:00-04:00
496,AA,7597.300000,63.350,63.270,63.360,63.270,1777648680000000000,161,2026-05-01 11:18:00-04:00
497,AA,1445.072000,63.310,63.340,63.340,63.310,1777648740000000000,51,2026-05-01 11:19:00-04:00
498,AA,5736.958118,63.350,63.400,63.420,63.350,1777648800000000000,87,2026-05-01 11:20:00-04:00
